# <font color="blue"> **Graph and Dynamical Systems Approaches to Whole-Brain Neuronal Networks** </font>
## <font color="blue"> **Structural and Functional Connectivity** </font>
<figure>
    <img src="https://mcgovern.mit.edu/wp-content/uploads/2021/10/brainmap_900x600.jpg" alt="Connectome" width="300"/>
</figure>


Notebook prepared for the:
<blockquote>

**Frontiers in Neurophotonics Summer School 2026**


2026 June 1 - 12, Quebec City, Canada
</blockquote>


**Author:**
* Patrick Desrosiers ([Dynamica Reaserch Lab](https://dynamicalab.github.io/) & [CERVO Brain Research Center](https://cervo.ulaval.ca/en/))


**Introduction:**

Network science has significantly advanced neuroscience by enabling the detailed analysis of brain connectivity through the use of mathematical models called graphs ([Bassett et al. 2017](https://doi.org/10.1038/nn.4502)).  These models have been pivotal in examining brain dynamics, neurodevelopment, and in the application of control theory to predict and influence brain state transitions, leading to potential new therapeutic strategies ([Barabási et al. 2023](https://doi.org/10.1523/JNEUROSCI.1014-23.2023)).

Here, using the AI collaborator Gemini and Python libraries like [NetworkX](https://networkx.org/), we will:
* explore graph representations,
* extract graph features,
* analyze structural connectivity data,
* extract and analyze functional connectivity data,
* compare functional and structural connectivity data.

### **Table of Contents**

<details>
<summary><strong>1. Graphs</strong></summary>

- 1.1 Definitions  
- 1.2 NetworkX  
  - 1.2.1 Visual display  
  - 1.2.2 Matrix Representation  
  - 1.2.3 Visualization of weighted and directed graphs  
  - 1.2.4 Random weighted and directed graphs  

</details>

<details>
<summary><strong>2. Structural Connectivity</strong></summary>

- 2.1 Relationship with graphs  
- 2.2 Connectomes at different scales  
- 2.3 The very first structural connectome: C. elegans  
- 2.4 Structural connectivity at mesoscale: Zebrafish  
  - 📁 Data Source  
  - Morphological data  
    - Projections  
    - Node positions  
    - Connectivity embedded in morphology  
    - Complementary information: Brain region names and indices  

</details>

<details>
<summary><strong>3. Graph Metrics</strong></summary>

- 3.1 Density  
- 3.2 Transitivity (global clustering coefficient)  
- 3.3 Degrees  
  - Degree-preserving random graphs
- 3.4 Further graph metrics
  - Shortest path length  
  - Network Efficiency
  - Small-world property  
  - Motifs  
  - Rich-Club Coefficient  
  - Modularity  
  - Singular Values and Effective Ranks  
  - Exercise: Mesoscale SC of the mouse brain  
  - Exercise: Microscopic SC of the larval drosophila  


</details>

<details>
<summary><strong>4. Functional Connectivity</strong></summary>

- 4.1 Relationship with Graphs  
- 4.2 Functional dataset from Paul De Koninck's Lab  
  - Experimental Setup  
  - Dataset: Activity of 70 Brain Regions in 22 Larval Zebrafish  
  - Extraction of Functional Connectomes  
  - Structure-Function Relationship  
  - Function-Distance Relationship  
- 4.3 Further analysis (optional)  

</details>

<details>
<summary><strong>5. Complementary References</strong></summary>

</details>


### **Python Environment Setup**

In [1]:
# Setup: install required Python packages (only needed once, e.g., in Google Colab) silently
!pip install -q networkx numpy scipy python-louvain seaborn pandas > /dev/null 2>&1

# Import required packages using standard abbreviations and print versions
# Load modules for computation and data analysis
import networkx as nx
import numpy as np
import scipy as sp
import pandas as pd
import community

print("networkx:", nx.__version__)
print("numpy:", np.__version__)
print("scipy:", sp.__version__)
print("pandas:", pd.__version__)
#print("python-louvain:", community.__version__)

# Load modules for plotting and visualization
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
print("matplotlib:", matplotlib.__version__)
print("seaborn:", sns.__version__)

networkx: 3.6.1
numpy: 2.0.2
scipy: 1.16.3
pandas: 2.2.2
matplotlib: 3.10.0
seaborn: 0.13.2


## <font color="blue"> **1. Graphs** </font>

### <font color="black"> **1.1 Definitions** </font>

A graph is a mathematical structure used to model pairwise relations between objects. It is composed of two main components:

1. **Vertices (or Nodes):** These are the fundamental units or points in the graph. Each vertex can represent various entities such as people in a social network, web pages on the internet, or neurons in a brain network.

2. **Edges (or Links):** These are the connections between pairs of vertices. An edge can represent a relationship, interaction, or a direct connection between two vertices. Edges can be undirected (no direction, indicating a mutual relationship) or directed (with direction, indicating a one-way relationship).

<figure>
    <img src="https://networkx.org/documentation/stable/_images/sphx_glr_plot_simple_graph_001.png" alt="Simple graph" width="300"/>
    <figcaption>Figure 1: Simple Graph with $N=5$ vertices, $M=6$ edges, and density 0.6.  </figcaption>
</figure>

Graph theory provides a rich set of tools and metrics to analyze these networks, such as connectivity patterns, network robustness, efficiency of information transfer, and centrality measures. This analysis helps to uncover the underlying architecture of brain networks and their role in neural function.

One of the basic **features (properties or metrics)** of a graph is its overall connectivity or **density**, measured as the ratio of the number of edges present to the number of possible edges between nodes in the graph:

$$
\text{density} = \frac{\text{number of edges}}{\text{number of possible edges}}=\frac{2M}{N(N-1)},
$$

where $M$ and $N$ denote the number of edges and nodes, respectively

---

#### Types of Graphs

- **Undirected Graph:** A graph in which edges have no direction. If vertex A is connected to vertex B, then vertex B is also connected to vertex A.
- **Directed Graph (Digraph):** A graph in which edges have a direction. An edge from vertex A to vertex B does not imply an edge from vertex B to vertex A.
- **Weighted Graph:** A graph in which each edge has a numerical value (weight) associated with it. This can represent distances, costs, or other quantities.
- **Unweighted Graph:** A graph in which all edges are considered equal, with no specific weight assigned to them.

<figure>
    <img src="https://raw.githubusercontent.com/pdesrosiers/public_data/main/images/graph_types.png" alt="Connectome" width="800"/>
</figure>


For simplicity, we will first focus on undirected graphs, starting with unweighted graphs, and then introduce weights and directed edges.

---

##### 💻 **Task:**

> Ask Gemini whether it agrees with the content of the cell above ("1.1 Definitions") and to provide its assessment in a separate Markdown cell.

----

Yes, I agree with the content of the '1.1 Definitions' cell. It accurately describes the fundamental components of a graph (vertices and edges), introduces basic graph properties like density, and correctly differentiates between common types of graphs: undirected, directed, weighted, and unweighted. The formulas and examples provided are also standard in graph theory.

### **1.2 NetworkX**
NetworkX is a Python library used for the creation, manipulation, and study of complex networks and graphs. It provides tools for analyzing the structure and dynamics of networks, offering various algorithms for network analysis and a wide range of built-in graph layouts for visualization.

---

##### 💻 **Task:**

>Ask Gemini to generate Python code using NetworkX that creates a graph with 5 nodes and 6 edges, and prints the number of nodes and the number of edges.

>Once the code has been generated, ask Gemini to execute it and verify the output.

>Finally, ask Gemini to explain the code step by step in clear language.

---

---

##### 💻 **Task:**

> Modify the graph above by adding **three new edges**:
> * One edge that connects two **existing nodes** already in the graph.
> * Two edges that introduce **new nodes** (nodes that don't currently exist in the graph).
>
> Then, print the updated number of **nodes** and **edges**.

---

#### **1.2.1 Visual display**

NetworkX offers many possibilities for displaying graphs, each with its own advantages and limitations.

- **Random Layout**: Positions nodes randomly in the graph space, providing a quick and unstructured visualization. *Advantage*: Simple and fast to compute. *Limitation*: Can be cluttered and lacks meaningful structure.
- **Circular Layout**: Arranges nodes in a circle, highlighting the overall structure and making it easy to identify certain types of symmetries and clusters. *Advantage*: Clearly shows node distribution and symmetrical patterns. *Limitation*: Can be less informative for large or complex networks.
- **Force-Directed (Spring) Layout**: Models the graph using repulsive forces between nodes and attractive forces along edges, similar to charged particles and springs. Nodes are iteratively adjusted until the system reaches equilibrium, minimizing edge crossings and distributing nodes evenly. *Advantage*: Provides an intuitive visualization that highlights clusters and relationships. *Limitation*: Computationally intensive for large graphs.


---

##### 💻 **Task:**

> Ask Gemini to generate three layounts of the previously defined graph using NetworkX functions `nx.random_layout`, `nx.circular_layout`,`nx.spring_layout`,`nx.draw_networkx_nodes` , `nx.draw_networkx_edges`.

---

---

##### 🚀 **Bonus Exercise:**

> Modify the code above so that:
>
> * the nodes are drawn as **red circles** using `node_color='red'`,
> * the node borders are **white**,
> * the edges are **blue**.
>
> Run the modified code and observe how these style changes affect the graph's appearance. Then, ask Gemini to explain which drawing options were changed and what visual effect each option has.

---


#### **First Random Graphs**

Random graphs are fundamental tools in network science. They serve as essential benchmarks for statistical analysis—allowing us to compare real-world networks against a baseline of randomness—and are incredibly useful for writing and testing code with synthetic data.

Below, we will randomly generate a graph with $N=12$ nodes and $M=28$ edges using `nx.gnm_random_graph` and then visualize it in three different ways. Each time you run the code, you will get a different realization of the network.

---

##### 💻 **Task:**

> Ask Gemini to write a Python script that uses NetworkX to generate a random graph with 12 nodes and 28 edges using `nx.gnm_random_graph`, and have it plot the result.
>
> **Run the code cell multiple times.** Observe how the network structure and layout change with each execution, demonstrating that the graph is truly generated randomly.

---

#### **1.2.2 Matrix Representation**

An **adjacency matrix** is a square matrix used to represent a graph. The elements of the matrix indicate whether pairs of vertices are adjacent or not in the graph. Given a graph $G$ with $N$ vertices, its adjacency matrix $A$ is an $N \times N$ matrix whose entry in the $i$-th row and $j$-th column, denoted $A_{ij}$, is defined as:
$$
A_{ij} =
\begin{cases}
1 & \text{if there is an edge between vertex } i \text{ and vertex } j, \\
0 & \text{otherwise.}
\end{cases}
$$
For undirected graphs, since the edge between vertex $i$ and vertex $j$ is the same as between vertex $j$ and vertex $i$, the matrix is symmetric:
$$
A_{ij} = A_{ji}.
$$

For instance, the adjacency matrix corresponding to the graph in Figure 1 is written as:
$$
A = \begin{pmatrix}
0 & 1 & 1 & 0 & 1 \\
1 & 0 & 1 & 0 & 0 \\
1 & 1 & 0 & 1 & 0 \\
0 & 0 & 1 & 0 & 1 \\
1 & 0 & 0 & 1 & 0 \\
\end{pmatrix}
$$
This matrix shows a '1' wherever there is a direct connection between nodes, and '0' otherwise. The matrix is symmetric, indicating that the graph is undirected.

---

##### 💻 **Task:**

> Ask Gemini to write a Python script using NetworkX to get the adjacency matrix representation of your previous random graph and print it out.

---

---

##### 💻 **Task:**

> Ask Gemini to modify your previous Python script to assign a random weight to each edge, turning it into a weighted graph.
>
> Have it extract and print the new **weighted adjacency matrix**, and verify that the matrix remains symmetric.

---

---

##### 💻 **Task:**

> Printing a raw matrix with numbers can be hard to read as the network grows. In neuroscience, connection strengths are typically visualized using a color-coded grid.
>
> Ask Gemini to use a plotting library like `seaborn` (with `sns.heatmap`) or `matplotlib` (with `plt.imshow`) to display your weighted adjacency matrix as a **heatmap**.
>
> *Hint: Make sure it includes a colorbar so you can interpret the connection strengths!*

---

#### **Optional: Generating Graphs from Matrices**

Just as we can extract an adjacency matrix from an existing NetworkX graph, we can also perform the reverse operation: generating a complete graph directly from a matrix. This is highly useful when importing structural or functional brain connectivity data that has already been preprocessed into matrix form.

Whether the matrix contains binary values (0 and 1) or continuous weights, NetworkX can reconstruct the underlying network topology seamlessly.

---

##### 🚀 **Bonus Exercise:**

> Ask Gemini to write a Python script that takes the weighted adjacency matrix from your previous step, uses NetworkX to rebuild a new graph from it, and verifies that this new graph has the exact same number of nodes and edges as your original random graph.

---

##### 🚀 **Bonus Exercise:**

> When we reconstruct a graph from its matrix, the network structure should be completely identical to the original. In network science, when two graphs share the exact same structural topology, they are said to be **isomorphic**.
>
> Ask Gemini to find the appropriate NetworkX function to programmatically check if your original random graph and the newly reconstructed matrix graph are actually **isomorphic**.

---

#### **1.2.3 Visualization of Weighted Graphs**

When analyzing brain networks, structural or functional connections often vary in strength. To make our network visualizations more intuitive, we can map these edge weights directly to the visual thickness of the lines.

Below, we will return to our weighted random graph example and use NetworkX to adjust the edge widths dynamically based on their assigned values.

---

##### 💻 **Task:**

> Ask Gemini to write a Python script that plots your weighted random graph, making the **thickness of each edge proportional to its weight**.
>
> *Hint: Ask Gemini to use the `width` parameter inside the `nx.draw` or `nx.draw_networkx_edges` function.*

---

#### **1.2.4 Optional: Random Weighted and Directed Graphs**

Below, we will **randomly generate a directed graph** with a given number of vertices and edges, and then randomly attribute a continuous weight to each edge.

The `nx.gnm_random_graph(N, M)` model in NetworkX handles this by:
* Creating a network with exactly **$N$ nodes** and **$M$ edges**.
* Selecting **$M$ random pairs of nodes** to connect.
* Allowing the graph to be **directed** by simply passing the parameter `directed=True`.

---

##### 🚀 **Bonus Challenge:**

> Ask Gemini to generate a **directed, weighted random graph** with 12 nodes and 28 edges.
>
> Have it extract the adjacency matrix and print it. Observe the matrix: Is it still symmetric? Why or why not?

---

##### 🚀 **Advanced Bonus Challenge:**

> Visualizing directed graphs can be tricky because connections are one-way.
>
> Ask Gemini to plot your directed network using `nx.draw`, ensuring that the **arrows indicating edge direction** are clearly visible and that the node labels are easy to read.

---

## <font color="blue"> **2. Structural Connectivity** </font>

Structural connectomes are comprehensive maps that depict the physical or anatomical connections between different elements within the brain, whether they are neurons, groups of neurons, or large brain regions. These maps detail the neural pathways and synapses, providing a structural framework for understanding how different parts of the brain are wired and how they communicate with each other.

### **2.1 Relationship with graphs**
Structural connectomes are commonly represented as **graphs** in the field of network neuroscience. In this representation:

- **Vertices** (or nodes) represent the structural components of the brain, such as individual neurons, clusters of neurons, or whole brain regions, depending on the scale of the connectome.
- **Edges** (or links) represent the physical connections between these nodes, such as axonal pathways or white matter tracts.

### **2.2 Connectomes at different scales**

Structural connectomes exist at different scales of observation: microscopic connectomes detail individual neurons and their synapses to reveal intricate brain architecture, mesoscopic connectomes examine groups of neurons within specific regions to explore local network interactions, and macroscopic connectomes focus on major brain areas, analyzing how these regions are interconnected to support complex behaviors and cognitive functions.

1. **Microscopic Structural Connectomes**:
   - **Scale**: Individual neurons and their synaptic connections.
   - **Graph Representation**: Each node represents a single neuron, and each edge represents a synaptic connection between neurons.
   - **Purpose**: To explore the most detailed level of brain architecture and understand the direct synaptic pathways that underpin specific neural circuits and their functionalities.
   - **Species studied**: Caenorhabditis elegans ([White et al. 1986](https://doi.org/10.1098/rstb.1986.0056)),  Ciona intestinalis ([Ryan et al. 2016]( https://doi.org/10.7554/eLife.16962)), Platynereis dumerilii ([Verasztó et al. 2020](https://doi.org/10.1101/2020.08.21.260984)), Drosophila melanogaster([Scheffer et al. 2020](https://doi.org/10.7554/eLife.57443), [Winding et al. 2023](https://www.science.org/doi/10.1126/science.add9330), [Lin et al. 2024](https://www.biorxiv.org/content/10.1101/2023.07.29.551086v2)).

2. **Mesoscopic Structural Connectomes**:
   - **Scale**: Groups of neurons or neural circuits within specific brain regions.
   - **Graph Representation**: Nodes represent groups of neurons or neural circuits, and edges represent the neural pathways (e.g., bundles of axons) connecting these groups.
   - **Purpose**: To study how local networks interact within larger brain systems and to map the connectivity within specific functional areas or along specific sensory/motor pathways.
   - **Species studies**: Mouse ([Oh et al. 2014](https://doi.org/10.1038/nature13186)), Zebrafish ([Kunst et al. 2019](https://doi.org/10.1016/j.neuron.2019.04.034)), etc.

3. **Macroscopic Structural Connectomes**:
   - **Scale**: Major brain regions or areas.
   - **Graph Representation**: Nodes are large brain regions (like the cortex, thalamus, etc.), and edges represent major white matter tracts that connect these regions.
   - **Purpose**: To understand the high-level organization of the brain and how broad regions collaborate to support complex behaviors and cognitive processes.
   - **Species studied**: Human, primates, other mammels, etc.

Each level of these connectomes provides unique insights into the brain's structure and functionality, offering different perspectives from the finest details to the broadest interactions.

---

##### 🌐 **Optional Task: Exploring Real Connectomes**

> Visit the **Netzschleuder network catalogue, repository, and centrifuge**:
>
> https://networks.skewed.de/
>
> Search for the keyword `connectome` and explore a few of the available brain networks.
>
> You may also consult the following curated collection of public brain network datasets:
>
> https://github.com/faskowit/brain-networks-across-the-web
>
> Briefly describe one connectome dataset that you find interesting and explain what its nodes and edges represent.

---

### **2.3 Optional: The very first structural connectome: C. elegans**

The first complete connectome was mapped for *Caenorhabditis elegans* (C. elegans), a transparent roundworm used as a model organism in neurobiology. The project was led by John White, part of Sydney Brenner's team, who published their findings in 1986. This work detailed the synaptic connections of all 302 neurons in C. elegans, providing crucial insights into neural networks and behavior. The simplicity and genetic tractability of C. elegans make it ideal for such detailed study, setting a precedent for further connectome research in more complex organisms.

<img src="https://upload.wikimedia.org/wikipedia/commons/4/46/Caenorhabditis_elegans_hermaphrodite_adult-en.svg" alt="Caenorhabditis elegans hermaphrodite adult" width="300"/> ![Moving C. elegans](https://upload.wikimedia.org/wikipedia/commons/b/be/CrawlingCelegans.gif)


---

##### 💻 **Task: Downloading and Visualizing the *C. elegans* Connectome**

> Ask Gemini to write Python code that connects directly to the following ZIP file:
>
> `https://networks.skewed.de/net/celegansneural/files/celegansneural.csv.zip`
>
> The code should:
>
> 1. use `requests` to download the ZIP file,
> 2. use `zipfile` and `io` to open the downloaded file,
> 3. extract the file `edges.csv`,
> 4. read the edge list into a Pandas DataFrame,
> 5. clean the column names if necessary,
> 6. build the corresponding **directed graph** in NetworkX,
> 7. print the number of nodes and the number of edges,
> 8. produce a visual representation of the graph.
>
> Once the code has been generated, run it and ask Gemini to explain each step.

---

After visualizing this connectome, we can appreciate the complexity and richness of its structure. Visualization provides an intuitive understanding of the network's architecture, revealing clusters, hubs, and potentially pathways. However, to systematically analyze and understand the functional and structural properties of this neural network, we need to move beyond visual inspection.

### **2.4 Structural connectivity at mesoscale: Zebrafish**

The larval zebrafish, typically around 5 to 7 days post-fertilization, is an ideal model organism for brain studies due to its small size and nearly transparent body, which allows for the imaging of its entire brain. Its brain comprises about 100,000 neurons, making it feasible to monitor all neurons under a microscope while the animal processes sensory information and generates behavior.

<img src="https://webhome.phy.duke.edu/~hsg/513/images/zebrafish-larva.jpg" width="150"/>

Herwig Baier's lab at the Max Planck Institute for Biological Intelligence (Martinsried, close to Munich) has achieved significant milestones in mapping a part of the brain's structural connectivity. Using a variety of techniques, the researchers genetically labeled individual neurons with fluorescent markers. This allowed them to visualize and digitize the shape and location of over 2,000 neurons, which were then registered to a standard brain model with 72 distinct brain regions.

<img src="https://www.cell.com/cms/attachment/8616d45e-8c5a-4508-9c74-5a28cf368a99/fx1.jpg" width="350"/>

The structural data are periodically updated in the [mapZbrain atlas](https://mapzebrain.org/home), now including more than 4400 single-cell tracings. Paul De Koninck's lab has used the latest connectomics data to create a mesoscale connectome, which represents the normalized number of connections between each pair of the 70 brain regions defined in the atlas.

References:
* [Kunst, M., Laurell, E., Mokayes, N., Kramer, A., Kubo, F., Fernandes, A. M., ... & Baier, H. (2019). A cellular-resolution atlas of the larval zebrafish brain. Neuron, 103(1), 21-38.](https://doi.org/10.1016/j.neuron.2019.04.034)
* [Légaré, A., Lemieux, M., Desrosiers, P., & De Koninck, P. (2023). Zebrafish brain atlases: a collective effort for a tiny vertebrate brain. Neurophotonics, 10(4), 044409-044409.](https://doi.org/10.1117/1.NPh.10.4.044409)
* [Légaré, A., Lemieux, M., Boily, V., Poulin, S., Légaré, A., Desrosiers, P., & De Koninck, P. (2025). Structural and genetic determinants of zebrafish functional brain networks. Science Advances, 11(28), eadv7576.](https://doi.org/10.1126/sciadv.adv7576)

#### 📁 Data Source

All structural data used in this notebook were originally published in:

**Légaré, Antoine; Lemieux, Mado; Boily, Vincent; Poulin, Sandrine; Légaré, Arthur; Desrosiers, Patrick; De Koninck, Paul**, 2025.  
*Structural and genetic determinants of zebrafish functional brain networks*.  
Borealis, V1. [https://doi.org/10.5683/SP3/IIVGOB](https://doi.org/10.5683/SP3/IIVGOB)

A curated subset of this dataset, necessary to run this notebook, is available here:  
🔗 [https://github.com/pdesrosiers/brain-connectivity-and-dynamics/tree/main/data/zebrafish](https://github.com/pdesrosiers/brain-connectivity-and-dynamics/tree/main/data/zebrafish)


---

##### 💻 **Task: Download the Zebrafish Connectome Dataset**

> Ask Gemini to write Python code that downloads the zebrafish dataset required for this notebook from the GitHub repository:
>
> `https://github.com/pdesrosiers/brain-connectivity-and-dynamics/tree/main/data/zebrafish`
>
> The code should:
>
> 1. use the `urllib.request` module,
> 2. download the following files:
>    * `SC_undirected.npy`
>    * `region_centroids.npy`
>    * `region_names.npy`
>    * `side_projection.npy`
>    * `top_projection.npy`
>    * `timeseries_regions_fish6.npy`
> 3. save the files locally,
> 4. display a success or error message for each download.


---

---

##### 💻 **Task: Build a Graph from the Zebrafish Connectivity Matrix**

> Ask Gemini to write Python code that:
>
> 1. loads the file `SC_undirected.npy`,
> 2. extracts the adjacency matrix stored in the file,
> 3. creates the corresponding graph in NetworkX,
> 4. computes and prints:
>    * the number of nodes,
>    * the number of edges,
> 5. displays the first few rows and columns of the adjacency matrix to verify that it was loaded correctly.

---

#### **Visualize the Structural Connectome (SC)**

---

##### 💻 **Task: Visualizing the Structural Connectome**

> Ask Gemini to write Python code that visualizes the structural connectome in two complementary ways:
>
> 1. Display the adjacency matrix as a **heatmap**, where color intensity represents the strength of the connection between pairs of brain regions.
> 2. Display the corresponding **weighted graph**, where edge thickness is proportional to the connection weight.
>
> Once the code has been generated:
>
> * run it,
> * inspect both visualizations,
> * and ask Gemini to explain how each visualization represents the same network information.

---

---

#### 🧠 Embedding the Connectome in Brain Morphology

So far, the structural connectome has been represented either as an adjacency matrix or as an abstract graph. However, brain networks are not only collections of nodes and edges: they are embedded in physical space.

In this section, we use anatomical projection images of the zebrafish brain together with the spatial coordinates of brain regions. This allows us to place the network directly onto a morphological representation of the brain. Each node corresponds to a brain region, and each edge represents a structural connection between regions.

---

##### 💻 **Task 1: Load and Prepare the Morphological Data**

> Ask Gemini to write Python code that:
>
> 1. loads the files `top_projection.npy` and `side_projection.npy`,
> 2. loads the file `region_centroids.npy`,
> 3. extracts the 2D coordinates needed to display the first 70 brain regions:
>    * `xy` coordinates for the top projection,
>    * `zy` coordinates for the side projection,
> 4. creates two dictionaries of node positions:
>    * `xy_dic` for the top view,
>    * `zy_dic` for the side view,
> 5. applies the necessary rotation and vertical flip to the side projection so that it is displayed in the correct orientation.

---

---

##### 💻 **Task 2: Visualize the Connectome on Brain Projections**

> Ask Gemini to write Python code that overlays the structural connectome onto the zebrafish brain morphology.
>
> The code should:
>
> 1. use the graph `G` created in the previous section,
> 2. draw the **side projection** of the brain and place the network nodes using `zy_dic`,
> 3. draw the **top projection** of the brain and place the network nodes using `xy_dic`,
> 4. draw the nodes as white circles with black outlines,
> 5. draw the edges with thickness proportional to their weights,
> 6. color the edges according to their connection strength,
> 7. add a colorbar indicating the edge weights.

---

---

##### 🚀 **Optional Exercise: Exploring the Brain in Three Dimensions**

> The previous visualizations projected the zebrafish brain onto two-dimensional views. However, the anatomical coordinates of the brain regions are inherently three-dimensional.
>
> Ask Gemini to write Python code that:
>
> 1. creates a **3D scatter plot** of the brain region centroids stored in `region_centroids.npy`,
> 2. displays each brain region as a point in 3D space,
> 3. uses different colors for the left and right hemispheres,
> 4. labels a subset of regions with their numerical indices,
> 5. adds appropriate axis labels and a title.

---

---

##### 🚀 **Optional Exercise: Identifying Brain Regions**

> So far, brain regions have been represented by numerical node identifiers. In practice, each node corresponds to a specific anatomical region of the zebrafish brain. To better understand the relationship between node IDs and brain anatomy, ask Gemini to write Python code that:
>
> 1. loads the file `region_names.npy`,
> 2. prints the correspondence between each region ID and its anatomical name,
> 3. displays the top and side brain projections,
> 4. overlays the brain-region centroids on the projections,
> 5. labels each node with its numerical region ID,
> 6. increases the node size to make the labels easier to read.

---

---

#### 🧠 Distance Constraints in Brain Networks

Building and maintaining long-range connections is biologically costly. As a result, brain networks are often organized according to a trade-off between minimizing wiring cost and maximizing communication efficiency.

Using the spatial coordinates of the zebrafish brain regions, we can investigate how connection strength depends on physical distance. Do nearby regions tend to be more strongly connected than distant regions? If so, what mathematical relationship best describes this decay?

In this section, we combine the structural connectome with the three-dimensional positions of brain regions to explore how connectivity varies as a function of distance.

---

##### 💻 **Task: Does Connectivity Decay with Distance?**

> Ask Gemini to write Python code that:
>
> 1. computes the Euclidean distance between every pair of brain regions using the coordinates stored in `region_centroids.npy`,
> 2. extracts all existing structural connections from the adjacency matrix,
> 3. associates each connection with the physical distance separating the corresponding brain regions,
> 4. builds a weighted histogram showing the distribution of connection strengths as a function of distance,
> 5. fits several candidate models to the data:
>    * an exponential decay,
>    * a log-normal distribution,
>    * a gamma distribution,
> 6. computes the coefficient of determination ($R^2$) for each fit,
> 7. displays the fitted curves together with the empirical distribution.

---

## <font color="blue"> **3. Graph Metrics** </font>

In network neuroscience, graph theory metrics (topological features or statistics) are essential for analyzing the structural and functional properties of neural networks. Below, we will explore the most commonly used metrics ([Rubinov & Sporns, 2010](https://doi.org/10.1016/j.neuroimage.2009.10.003), [Basset et al. 2017](https://doi.org/10.1038/nn.4502), [Zamani Esfahlani et al. 2022](https://doi.org/10.1038/s41467-022-29770-y)).

We now review some of the most relevant graph metrics and use them to characterize the C. elegans connectome.  

### **3.1 Density**

The density (a.k.a. connectectivity or connection probability $p$) of a graph is a measure of how many edges are present relative to the total number of possible edges between its nodes.

In [2]:
density = nx.density(G)
print(f"Density: {density:.3f}")

NameError: name 'G' is not defined

#### **More random graphs**

The **Erdős–Rényi (ER)** model is a fundamental model of random graphs where each possible edge between a pair of $ N $ nodes is created **independently** with a fixed **probability $ p $**. This version of the model is often referred to as **G(n, p)**.

- In **G(n, p)**, the **expected number of edges** is $ p \cdot \frac{N(N-1)}{2} $ for undirected graphs, or $ p \cdot N(N-1) $ for directed graphs.
- Every edge is formed independently, which means the total number of edges may vary between graph realizations.

This differs from the **G(n, m)** model, where exactly **$ m $** edges are added randomly to the graph. In G(n, m), the total number of edges is fixed in advance, while in G(n, p), the number of edges is a random variable centered around its expected value.

In our case, once the density of the empirical connectome is known, we can use it as the parameter $ p $ in the ER model to generate **random graphs with similar average sparsity**, allowing for useful comparisons with the real brain network.


In [ ]:
# Generate a random graph with the same number of nodes and density using ER model
N = G.number_of_nodes()  # Number of nodes in graph G
p = density  # Edge probability based on the density of G

# Generate randomly a grapg from the ER model
G_random = nx.erdos_renyi_graph(N, p, directed=False)

# Calculate density of the generated random graph (optional check)
density_random = nx.density(G_random)
print(f"Density of generated random graph G_random: {density_random:.3f}\n")

# Plotting both graphs side by side
fig, axs = plt.subplots(1, 2, figsize=(10, 5))

# Plot original graph G
axs[0].set_title('Original Graph G')
pos = nx.spring_layout(G, k=0.2, seed=42)  # Fixed seed and k for layout consistency
nx.draw(G, pos, with_labels=False, edge_color='grey', node_size=50, ax=axs[0],
        alpha=0.5)

# Plot random graph G_random
axs[1].set_title('Random Graph G_random')
pos_random = nx.spring_layout(G_random, k=0.2, seed=42)  # Fixed seed and k for layout consistency
nx.draw(G_random, pos_random, with_labels=False, node_color='lightgreen', edge_color='grey',
        node_size=50, ax=axs[1], alpha=0.5)

plt.tight_layout()
plt.show()

### **3.2 Transitivity (global clustering coefficient)**:

  - **Definition**: This metric measures the overall probability that the adjacent nodes of a node are connected, calculated as the ratio of the number of triangles to the number of connected triples of nodes.
  - **Relevance**: High transitivity indicates a strong tendency for nodes to form tightly knit groups, which can be critical for local information processing and robustness in neural networks.
  - **Expectation**: High transitivity is expected in brain connectomes, reflecting the presence of tightly knit groups or clusters that contribute to local processing and network robustness.
  - **Formula**: In an undirected graph, the global clustering coefficient is calculated as:
  $$
    C = \frac{3 \times \text{number of triangles}}{\text{number of connected triples of nodes}}
  $$

<figure>
    <img src="https://raw.githubusercontent.com/pdesrosiers/public_data/main/images/clustering.png" alt="Clustering" width="800"/>
</figure>

---

##### 💻 **Task: Computing the Transitivity of the Connectome**

> Write Python code that:
>
> 1. computes the transitivity of the graph `G` using `nx.transitivity()`,
> 2. stores the result in a variable named `transitivity`,
> 3. prints the value of the transitivity.

---

##### 💻 **Task: Comparing the Connectome to Random Graphs**

> Write Python code that:
>
> 1. generates a random Erdős–Rényi graph with the same number of nodes and density as the connectome,
> 2. computes its transitivity,
> 3. compares this value to the transitivity of the connectome.
>
> Generate 1000 Erdős–Rényi random graphs with the same number of nodes and density as the connectome. Compute the fraction of random graphs whose transitivity is greater than or equal to that of the connectome.

---

### **3.4 Degrees**

<figure>
    <img src="https://raw.githubusercontent.com/pdesrosiers/public_data/main/images/degree_and_distribution.png" alt="Clustering" width="600"/>
</figure>

   - **Definition**: The degree of a node is the number of edges connected to it. In directed graphs, this can be divided into in-degree (incoming connections) and out-degree (outgoing connections).
   - **Relevance**: High-degree nodes may serve as important hubs in neural networks, potentially playing critical roles in neural processing and information dissemination.
   - **Expectation**: In brain connectomes, the degree distribution is often heterogeneous, with a **few nodes having very high degrees (hubs) and many nodes having low degrees**. This heterogeneity reflects the presence of hub nodes that are central to network communication and integration.


<figure>
    <img src="https://raw.githubusercontent.com/pdesrosiers/public_data/main/images/connectome_degree_distribution.png" alt="Clustering" width="600"/>
</figure>

[Giacopelli et al. 2021](https://www.nature.com/articles/s41598-021-83759-z)




##### 💻 **Task: Visualizing Degree and Strength Distributions**

> Write Python code that:
>
> 1. computes the degree of every node in the graph `G`,
> 2. computes the strength (weighted degree) of every node,
> 3. stores the results in variables named `degree_sequence` and `strength_sequence`,
> 4. plots a histogram of the degree distribution,
> 5. plots a histogram of the strength distribution,
> 6. displays both histograms side by side.

---

---

##### 🚀 **Optional Exercise: Descriptive Statistics and Normality Testing**

> Write Python code that:
>
> 1. computes the mean, median, standard deviation, minimum, and maximum of the `degree_sequence`,
> 2. computes the same statistics for the `strength_sequence`,
> 3. performs a Shapiro–Wilk test on both distributions using `scipy.stats.shapiro`,
> 4. reports the test statistics and associated p-values.

---

#### **Degree-preserving random graphs**

Once the degree sequence is known, we can use it to sample **null models**—random graphs that preserve key properties of the observed network, allowing us to test whether other statistics (e.g., clustering, efficiency) are **significantly different** from what would be expected by chance.

This is especially important in neuroscience and other biological domains, where some structural features (like degree heterogeneity) are known to influence many graph measures.

Two common models are:

- **Configuration Model**  
  Generates random graphs that exactly match a given **degree sequence**, but otherwise connect edges randomly. This allows one to isolate the effect of the degree distribution on network measures, without enforcing other properties like clustering or modularity.

- **Directed Configuration Model**  
  Extends the above model to **directed graphs**, preserving both **in-degree** and **out-degree** sequences. This is crucial when directionality carries functional or causal significance (e.g., in neural circuits).


---

##### 💻 **Task: Generating a Degree-Preserving Random Graph**

> Write Python code that:
>
> 1. extracts the degree sequence of the graph `G`,
> 2. generates a random graph using the configuration model,
> 3. converts the result into a simple graph by removing parallel edges and self-loops,
> 4. computes independent layouts for the original and randomized graphs,
> 5. displays the original graph and the degree-preserving random graph side by side.

---  

---

##### 💻 **Task: Testing Transitivity with a Degree-Preserving Null Model**

> Write Python code that:
>
> 1. generates 1000 random graphs using the configuration model,
> 2. computes the transitivity of each random graph,
> 3. stores the values in a list named `null_transitivities`,
> 4. plots the distribution of `null_transitivities`,
> 5. adds a red vertical line indicating the observed transitivity of the connectome,
> 6. estimates the p-value as the proportion of random graphs with transitivity greater than or equal to the observed value.

---


### **3.4 Further graph metrics**

#### **3.4.1 Shortest path length**
   - **Definition**: The path length is the number of edges in the shortest path between two nodes. The average shostest path length (SPL) is the average of these shortest paths taken over all pairs of nodes in the graph.
   - **Relevance**: This measure helps in understanding the efficiency of information transfer across the network. Shorter average path lengths typically indicate more efficient global communication.
   - **Expectation**: In brain connectomes, shorter average path lengths are expected, indicating efficient communication pathways that facilitate rapid information transfer between distant brain regions.

"The figure below is a graph of $ N=8 $ nodes and $ M=13 $ edges. The purple lines indicate a possible path between nodes 1 and 8, while the magenta lines show the shortest path between these nodes, with a length of 4.

<figure>
    <img src="https://raw.githubusercontent.com/pdesrosiers/public_data/main/images/shortest-path-length.png" alt="Clustering" width="600"/>
</figure>

In a directed graph, the **average shortest path length** can only be computed inside a strongly connected component (SCC), which is subset of nodes where each node is reachable from every other node within this subset, considering the direction of the edges. Formally, a SCC in a directed graph $ G = (V, E) $ is a maximal subgraph $ G' = (V', E') $ such that for every pair of vertices $ u, v \in V' $, there exists a directed path from $ u $ to $ v $ and from $ v $ to $ u $ in $ G' $.

In [ ]:
# Check if the graph is connected
if nx.is_connected(G):
    print("The graph is connected.")

    # Compute the average shortest path length
    avg_path_length = nx.average_shortest_path_length(G)
    print(f"Average shortest path length: {avg_path_length:.3f}")
else:
    print("The graph is not connected. You should extract the largest connected component using:")
    print("- nx.connected_components(G) to get all components")
    print("- max(..., key=len) to find the largest one")
    print("- G.subgraph(...) to create the subgraph")
    print("- Then compute nx.average_shortest_path_length(...) on this subgraph")

To compute the average shortest path length while taking edge weights into account, simply pass the weight parameter like below. This treats the weights as distances when calculating shortest paths.

In [ ]:
# Compute the average shortest path length
weighted_avg_path_length = nx.average_shortest_path_length(G, weight='weight')
print(f"Average shortest path length: {weighted_avg_path_length:.3f}")

<h4><font color="magenta">💻 <strong>Optional Exercise:</strong></font></h4>

>For directed graphs, analyze the structure of strongly connected components (SCCs).
>
> - Use nx.number_strongly_connected_components(G) to determine how many SCCs exist.
> - Identify and extract the largest SCC, and compute the average shortest path length within it.
> - Compute the average shortest path length for all SCCs with more than one node, and report the mean of these values.

<details>
<summary>✅ <strong>Solution</strong></summary>

```python

# Get the number of strongly connected components
num_scc = nx.number_strongly_connected_components(G)

print(f"Number of Strongly Connected Components: {num_scc}")

# Get all strongly connected components for directed graph (DiGraph)
components = list(nx.strongly_connected_components(G))

# Identify the largest strongly connected component
largest_scc = max(components, key=len)

# Create a subgraph for the largest strongly connected component
largest_scc_subgraph = G.subgraph(largest_scc)

# Number of nodes in the largest SCC
num_nodes_largest_scc = len(largest_scc)

print(f"Number of nodes in the largest SCC: {num_nodes_largest_scc}")

# Compute the average shortest path length within the largest SCC
if largest_scc_subgraph.number_of_nodes() > 1:
    avg_shortest_path_length = nx.average_shortest_path_length(largest_scc_subgraph)
    print(f"Average shortest path Length in the largest SCC: {avg_shortest_path_length:.3f}")
else:
    print("The largest SCC has only one node, so average shortest path length is not defined.")

# Initialize a list to store average shortest path lengths of each component
avg_shortest_path_lengths = []

# Iterate over each strongly connected component
for component in components:
    # Create a subgraph induced by the nodes in the component
    subgraph = G.subgraph(component)

    # Check if the component has more than one node
    if len(subgraph) > 1:
        # Compute all-pairs shortest path lengths in the subgraph
        shortest_path_lengths = dict(nx.shortest_path_length(subgraph))

        # Calculate the average shortest path length for this component
        avg_shortest_path_length = sum(
            sum(length for length in path_lengths.values()) / (len(subgraph) - 1)
            for path_lengths in shortest_path_lengths.values()
        ) / len(subgraph)

        # Append the average shortest path length to the list
        avg_shortest_path_lengths.append(avg_shortest_path_length)

# Compute the overall average shortest path length
if avg_shortest_path_lengths:
    overall_avg_shortest_path_length = sum(avg_shortest_path_lengths) / len(avg_shortest_path_lengths)
    print(f"Overall average shortest path length for strongly connected components: {overall_avg_shortest_path_length:.3f}")
else:
    print("No strongly connected components found.")
```

#### **Network Efficiency**:
   - **Definition**: Global efficiency is the average inverse shortest path length in the network, and local efficiency is the efficiency of the subgraphs formed by the neighbors of each node.
   - **Relevance**: Efficiency metrics help in understanding the functional integration and fault tolerance of neural networks.
   - **Expectation**: High global and local efficiency are expected in brain connectomes, reflecting the need for effective integration of information and resilience to local disruptions.
  - **Formula**: Global efficiency $ E $ is calculated as:
  $$
    E  = \frac{1}{N(N-1)} \sum_{i \neq j \in G} \frac{1}{d_{ij}}
  $$
  where $ d_{ij} $ is the shortest path length between nodes $ i $ and $ j $.

In [ ]:
# Compute global efficiency
global_eff = nx.global_efficiency(G)
print(f"Global Efficiency: {global_eff:.4f}")

#### **3.4.2 Small-world property**

Typically, connectomes are **small-world**. In Network Science, a small-world graph is characterized by **high clustering** (transitivity) and **low average shortest path length**."

The small-worldness coefficient, often denoted as $\sigma$, quantitatively measures whether a network exhibits small-world properties. It is defined as the ratio of the normalized clustering coefficient and the normalized average shortest path length of a network, compared to those of a random network with similar size and degree distribution:
$$ \sigma = \frac{C/C_{rand}}{L/L_{rand}} $$
where:
- $ C $ is the clustering coefficient of the network.
- $ C_{rand} $ is the average clustering coefficient of a set of random networks with the same number of nodes and edges as the original network.
- $ L $ is the average shortest path length of the network.
- $ L_{rand} $ is the average shortest path length of the corresponding random networks.

A value of $\sigma > 1$ suggests that the network exhibits small-world properties, indicating a balance between high local clustering and short global path lengths compared to random networks. Note that $\sigma$ can only be computed for undirected graphs, so if the graph is directed, it must be first symmetrized before using the function `nx.sigma`.

In [ ]:
# Function nx.sigma is too slow. New one defined below.
# # Create an undirected version of G
# G_undirected = nx.Graph(G)
# # Compute the small-worldness coefficient
# small_worldness = nx.sigma(G_undirected, niter=100, nrand=10, seed=None)
# # Print the result
# print(f"Small-worldness coefficient: {small_worldness}")


def small_worldness_coefficient(G, n_iter=32):
    """
    Compute the small-worldness coefficient sigma for a graph G.

    Parameters:
    - G: NetworkX graph
    - n_iter: Number of iterations for random graph generation

    Returns:
    - sigma: Small-worldness coefficient
    """
    # Warn and convert to undirected if needed
    if nx.is_directed(G):
        print("Warning: Graph was converted to undirected to compute small-worldness coefficient.")
        G = G.to_undirected()

    # Check if the graph is connected
    if not nx.is_connected(G):
        print("Graph is not connected. Computing for the largest connected component.")
        # Get the largest connected component
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()

    # Compute actual sigma
    n = G.number_of_nodes()
    m = G.number_of_edges()

    # Get degrees for the largest connected component
    degree_sequence = [d for n, d in G.degree()]

    # Calculate average shortest path length for the original graph
    L = nx.average_shortest_path_length(G)

    # Compute average shortest path length for configuration model instances
    L_random = 0
    for _ in range(n_iter):
        # Generate configuration model random graph
        G_random = nx.configuration_model(degree_sequence)

        # Convert to simple graph if G_random is a multigraph
        if G_random.is_multigraph():
            G_random = nx.Graph(G_random)

        # Check if the graph is connected
        if not nx.is_connected(G_random):
            print("Generated random graph is not connected. Skipping iteration.")
            continue

        L_random += nx.average_shortest_path_length(G_random) / n_iter

    # Calculate average clustering coefficient for the original graph
    try:
        C = nx.average_clustering(G)
    except nx.NetworkXError:
        print("Error computing average clustering coefficient for original graph.")
        C = 0

    # Compute average clustering coefficient for configuration model instances
    C_random = 0
    for _ in range(n_iter):
        # Generate configuration model random graph
        G_random = nx.configuration_model(degree_sequence)

        # Convert to simple graph if G_random is a multigraph
        if G_random.is_multigraph():
            G_random = nx.Graph(G_random)

        # Check if the graph is connected
        if not nx.is_connected(G_random):
            print("Generated random graph is not connected. Skipping iteration.")
            continue

        try:
            C_random += nx.average_clustering(G_random) / n_iter
        except nx.NetworkXError:
            print("Error computing average clustering coefficient for random graph.")
            continue

    # Compute sigma
    if L_random == 0 or C_random == 0:
        sigma = 0
    else:
        sigma = (C / C_random) / (L / L_random)

    return sigma


sigma = small_worldness_coefficient(G)
print(f"Small-worldness coefficient sigma: {sigma:.3f}")

#### **3.4.3 Motifs**

Detecting motifs in complex networks is challenging due to the computational complexity involved in searching for subgraph patterns efficiently. For more robust and scalable motif detection, alternative tools like `graph_tool` in Python offer advanced algorithms and optimizations beyond what is commonly available in NetworkX. Other packages like `igraph` and
`pymfinder` also provide efficient methods for motif discovery in large-scale networks.

#### **3.4.4 Rich-Club Coefficient**
- **Definition**: This measures the tendency of high-degree nodes to be more densely interconnected than expected by chance.
- **Relevance**: In neural networks, a high rich-club coefficient among high-degree nodes suggests a "rich-club" architecture where hub nodes form a cohesive group, crucial for global communication and integration of information.
- **Expectation**: A high rich-club coefficient is expected in brain connectomes, where highly interconnected hub nodes facilitate efficient global integration and coordination across the network.
- **Formula**: The rich-club coefficient is calculated as:
  $$
  \phi(k) = \frac{2E_{>k}}{N_{>k}(N_{>k} - 1)}
  $$
  where $ E_{>k} $ is the number of edges among nodes with degree greater than $ k $, and $ N_{>k} $ is the number of such nodes.

- **Normalization**: To determine whether the observed rich-club coefficient is statistically significant, it should be normalized against a null model. This involves comparing the rich-club coefficient of the actual network to that of a randomized network. The normalized rich-club coefficient is given by:
  $$
  \rho(k) = \frac{\phi(k)}{\phi_{\text{rand}}(k)}
  $$
  where $ \phi_{\text{rand}}(k) $ is the rich-club coefficient of the randomized network. A value of $\rho(k) > 1$ indicates that high-degree nodes are more interconnected than expected by chance, suggesting a significant rich-club organization.

- **Randomized Networks**: To create a meaningful null model, the network is randomized while preserving certain properties, such as the degree distribution. This can be done using methods like degree-preserving randomization or the configuration model. The configuration model generates a randomized network that maintains the same degree sequence as the original network by randomly rewiring edges. This ensures that the observed rich-club phenomenon is not merely a result of the degree distribution but reflects a higher-order organization of the network.

In [ ]:
def rich_club_normalized(G, num_randomizations=10):
    """
    Compute the normalized rich-club coefficient of a graph G.

    Parameters:
    - G: NetworkX graph
    - num_randomizations: Number of randomized graphs to generate for averaging

    Returns:
    - normalized_rich_club_coeff: Dictionary keyed by degree with normalized rich-club coefficient values
    """
    # Compute the rich-club coefficients for the original graph
    original_phi = nx.rich_club_coefficient(G, normalized=False)

    # Initialize a dictionary to store the average rich-club coefficients for the randomized graphs
    phi_rand_sum = {k: 0 for k in original_phi.keys()}

    for _ in range(num_randomizations):
        # Create a configuration model graph to preserve the degree distribution
        random_graph = nx.configuration_model([d for n, d in G.degree()])

        # Remove parallel edges and self-loops
        random_graph = nx.Graph(random_graph)
        random_graph.remove_edges_from(nx.selfloop_edges(random_graph))

        # Compute the rich-club coefficients for the randomized graph
        random_phi = nx.rich_club_coefficient(random_graph, normalized=False)

        # Accumulate the rich-club coefficients for averaging
        for k in random_phi.keys():
            phi_rand_sum[k] += random_phi[k]

    # Compute the average rich-club coefficients for the randomized graphs
    phi_rand_avg = {k: phi_rand_sum[k] / num_randomizations for k in phi_rand_sum.keys()}

    # Normalize the rich-club coefficients, avoiding division by zero
    normalized_rich_club_coeff = {}
    for k in original_phi.keys():
        if phi_rand_avg[k] != 0:
            normalized_rich_club_coeff[k] = original_phi[k] / phi_rand_avg[k]
        else:
            normalized_rich_club_coeff[k] = float('inf')  # or you can choose to skip these degrees

    return normalized_rich_club_coeff



# Step 1: Compute the Rich-Club Coefficients
G_undirected = G.to_undirected()
rich_club_coefficient = rich_club_normalized(G_undirected)

# Step 2: Plot the Rich-Club Coefficient
degrees = list(rich_club_coefficient.keys())
coefficients = list(rich_club_coefficient.values())

plt.figure(figsize=(5,3))
plt.plot(degrees, coefficients, marker='o', linestyle='-', color='orange')
plt.ylim(0,6)
plt.xlabel('Degree')
plt.ylabel('Normlized Rich-Club Coefficient')
plt.grid(True)
plt.show()

In [ ]:
# Step 3: Identify the first degree for which the coefficient is larger than
# the seleted threshold
threshold = 1.5
for degree, coefficient in zip(degrees, coefficients):
    if coefficient > threshold:
        print(f"The first degree for which the rich-club coefficient is larger than {threshold} is: {degree}")
        break


# Step 4: Identify Rich-Club Nodes
degree_threshold = degree # Select a degree threshold, for example, where the rich-club coefficient significantly increases
rich_club_nodes = [node for node, degree in dict(G.degree()).items() if degree >= degree_threshold]

print(f"\nRich-club nodes (degree >= {degree_threshold}):\n", rich_club_nodes)

# Step 5: Plot the original graph with spring layout and highlight the rich-club nodes
# Plot the graph with force-directed info
plt.figure(figsize=(6,6))
pos = nx.spring_layout(G, k=0.5)
nx.draw_networkx_nodes(G, pos, node_size=70, node_color='bisque', edgecolors='black', linewidths=1)
nx.draw_networkx_edges(G, pos, alpha=0.8, edge_color='grey', width=0.6)
nx.draw_networkx_nodes(G, pos, nodelist=rich_club_nodes, node_size=100, node_color='coral', edgecolors='black', linewidths=1)

plt.title('Rich-Club Nodes Highlighted')
plt.axis('off')
plt.show()


For a detailed analysis of the rich club phenomenon in the C. elegans connectome, refer to the comprehensive study by Towlson et al., [published in the Journal of Neuroscience in 2013](https://doi.org/10.1523/JNEUROSCI.3784-12.2013).

#### **3.4.5 Modularity**
   - **Definition**: Modularity quantifies the degree to which the network may be subdivided into clearly defined groups or communities.
   - **Relevance**: High modularity indicates a structure with dense connections between the nodes within modules but sparse connections between nodes in different modules, suggesting specialized processing within tightly knit groups.
   - **Expectation**: High modularity is expected in brain connectomes, where functional segregation into distinct modules supports specialized processing and cognitive functions.
   - **Community Detection Algorithms**: There are many community (module) detection algorithms used to identify these modules within a graph ([Fortunato & Hric 2016](https://doi.org/10.1016/j.physrep.2016.09.002), [Betzel 2023](https://doi.org/10.1016/B978-0-323-85280-7.00016-6)). Two popular ones are:
     - **Greedy Modularity Communities**: This algorithm iteratively merges nodes or smaller communities to maximize modularity, stopping when no further merging increases the modularity score. It can be computationally intensive for large networks.
     - **Louvain Algorithm**: This efficient, two-phase method first assigns each node to its own community and optimizes modularity by moving nodes between communities. It then aggregates the network based on these communities and repeats the process. The algorithm stops when modularity no longer increases. The Louvain algorithm is faster and more scalable, particularly for large networks, due to its hierarchical approach.

The modularity $ Q $ is given by the following equation:
$$ Q = \frac{1}{2m} \sum_{i,j} \left[ A_{ij} - \frac{k_i k_j}{2m} \right] \delta(c_i, c_j) $$
where:
- $ A_{ij} $ is the element of the adjacency matrix $A $.
- $k_i $ and $ k_j $ are the degrees (number of edges) of nodes $ i $ and $ j $, respectively.
- $ m $ is the total number of edges in the network.
- $ \delta(c_i, c_j) $ is the Kronecker delta, which is 1 if nodes $ i $ and $ j $ are in the same community $i.e., $ c_i = c_j $$, and 0 otherwise.
- $ c_i $ and $c_j $ are the communities to which nodes $ i $ and $ j $ belong.

In the code below, we generate and visualize two graphs with different levels of modularity using the Stochastic Block Model (SBM) implemented NetworkX, demonstrating how community structure affects modularity. SBM is a model for generating random graphs with a predefined community structure, where the probability of edge formation depends on the community membership of the nodes.

In [ ]:
# Two graph generated randomly using SBM
# Define block sizes
n=100
sizes = [50, 50]

# Low modularity SBM graph
p_low = 0.08  # Probability of edges within communities
q_low = 0.08  # Probability of edges between communities
low_modularity_graph = nx.stochastic_block_model(sizes, [[p_low, q_low], [q_low, p_low]])

# High modularity SBM graph
p_high = 0.2
q_high = 0.02
high_modularity_graph = nx.stochastic_block_model(sizes, [[p_high, q_high], [q_high, p_high]])

# Plotting
fig, axs = plt.subplots(1, 2, figsize=(10,5))

# Plot low modularity graph
pos = nx.spring_layout(low_modularity_graph)
nx.draw(low_modularity_graph, pos, ax=axs[0], with_labels=False, node_size=90, edge_color='grey', alpha=0.75)
axs[0].set_title(f'Low Modularity Graph')

# Plot high modularity graph
pos = nx.spring_layout(high_modularity_graph)
nx.draw(high_modularity_graph, pos, ax=axs[1], with_labels=False, node_size=90, edge_color='grey', alpha=0.75)
axs[1].set_title(f'High Modularity Graph')

plt.show()

We now test Louvain and Greedy algoritmms on these two synthetic graphs.

In [ ]:
from networkx.algorithms import community
import community.community_louvain as community_louvain #python-louvain package
import warnings

def visualize_modularity(G, title, ax, algorithm='greedy'):
    """
    Visualize the modularity and communities in a graph.

    Parameters:
    - G: NetworkX graph
    - title: str, the title for the graph
    - ax: Matplotlib axis for the graph visualization
    - algorithm: str, the algorithm to use for community detection ('greedy' or 'louvain')
    """
    if algorithm == 'greedy':
        # Identify communities using the greedy modularity communities algorithm
        communities = list(community.greedy_modularity_communities(G))
    elif algorithm == 'louvain':
        if G.is_directed():
            warnings.warn("The Louvain algorithm does not support directed graphs. Converting to undirected graph.")
            G = G.to_undirected()
        # Identify communities using the Louvain algorithm
        partition = community_louvain.best_partition(G)
        print(f"Number of communities: {len(set(partition.values()))}")
        # Create a dictionary to store the communities
        communities = {}
        for node, comm in partition.items():
            communities.setdefault(comm, []).append(node)
        communities = [frozenset(community) for community in communities.values()]
    else:
        raise ValueError("Unsupported algorithm. Use 'greedy' or 'louvain'.")

    # Compute modularity
    modularity = community.modularity(G, communities)
    print(f"Modularity for the {title} using {algorithm} algorithm: {modularity}\n")

    # Create a color map for the communities
    color_map = {}
    for i, community_set in enumerate(communities):
        for node in community_set:
            color_map[node] = i

    # Assign colors to nodes based on their community
    node_colors = [color_map[node] for node in G.nodes()]

    # Position nodes using the spring layout
    pos = nx.spring_layout(G, k=0.4)

    # Draw nodes with community colors
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=300, cmap=plt.cm.Oranges, node_color=node_colors, edgecolors='black')

    # Draw all edges
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='grey', alpha=0.4)

    # Draw node labels
    # nx.draw_networkx_labels(G, pos, ax=ax, font_size=10, font_color='black')

    ax.axis('off')  # Hide the axes

    return modularity, node_colors



# Create subplots with 1 row and 2 columns
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Visualize modularity and communities for the low modularity graph
modularity, node_colors = visualize_modularity(low_modularity_graph, 'Low modularity', ax=axes[0], algorithm='louvain')
axes[0].set_title('Low Modularity Graph')

# Visualize modularity and communities for the high modularity graph
modularity, node_colors = visualize_modularity(high_modularity_graph, 'High modularity', ax=axes[1], algorithm='louvain')
axes[1].set_title('High Modularity Graph')

# Adjust layout
plt.tight_layout()
plt.show()


We're ready to compute the modularity for the zebrafish SC.

In [ ]:
# Create subplots
fig, ax = plt.subplots(figsize=(6, 6))

# Visualize modularity and communities
modularity, node_colors = visualize_modularity(G, 'Zebrafish SC', ax=ax, algorithm='louvain')
plt.title('Zebrafish SC with Highlighted Communities')
plt.tight_layout()
plt.show()

#### **3.4.6 Singular Values and Effective Ranks**

As highlighted in a [recent study](https://www.nature.com/articles/s41567-023-02303-0), the singular values $\sigma_1,\ldots, \sigma_N$ of adjacency matrices in connectomics exhibit rapid decay, indicating a low effective rank.


Singular value decomposition (SVD) is used to compute these values in Python, typically through `numpy.linalg.svd`. The SVD equation is:
$$ A = U \Sigma V^T $$
where $A $ is the adjacency matrix, $ U $ is an orthogonal matrix of left singular vectors, $ \Sigma $ is a diagonal matrix of singular values $ \sigma_i $, and $V^T $is an orthogonal matrix of right singular vectors.

Below, we extract the adjacency matrix of graph $ G $ and compute its singular values. We compare these values with those obtained from randomly generated graphs that preserve the same degrees as $ G $ (using the configuration model). Our analysis reveals that while the singular values of the original graph decrease rapidly, they exhibit a distinct behavior compared to those of randomly generated graphs, highlighting the more complex structure inherent in the original graph.

In [ ]:
import numpy as np

# Step 1: Extract adjacency matrix
if nx.is_directed(G):
    adj_matrix = nx.to_numpy_array(G, weight='weight', dtype=np.float64)
else:
    adj_matrix = nx.to_numpy_array(G, weight=None, dtype=np.float64)

# Step 2: Compute singular values
singular_values = np.linalg.svd(adj_matrix, compute_uv=False)

# Step 3: Plot the scree plot
# Plotting
fig, axs = plt.subplots(1, 2, figsize=(10,5))
# 1st plot
axs[0].plot(range(1, len(singular_values) + 1), singular_values, marker='o', linestyle='-', color='b')
axs[0].set_title('Scree Plot')
axs[0].set_xlabel('Singular Value ID')
axs[0].set_ylabel('Singular Value')
# 2nd plot
max_singular_value = np.max(singular_values)
norm_singular_values = singular_values/max_singular_value
axs[1].plot(range(1, len(singular_values) + 1), norm_singular_values, marker='o', linestyle='-', color='b')
axs[1].set_title('Normalized Scree Plot')
axs[1].set_xlabel('Singular Value ID')
axs[1].set_ylabel('Normalized Singular Value')
plt.show()

In [ ]:
def generate_random_graph_svd(G, num_random_graphs=100):
    """Generate random graphs from configuration model and compute their singular values."""
    singular_values_list = []

    for _ in range(num_random_graphs):
        if nx.is_directed(G):
            # Extract in-degree and out-degree sequences
            in_degree_sequence = [d for n, d in G.in_degree()]
            out_degree_sequence = [d for n, d in G.out_degree()]

            G_random = nx.directed_configuration_model(in_degree_sequence, out_degree_sequence, create_using=nx.DiGraph())
        else:
            # Extract degree sequence for undirected graph
            degree_sequence = [d for n, d in G.degree()]

            G_random = nx.configuration_model(degree_sequence, create_using=nx.Graph())

        # Extract adjacency matrix
        adj_matrix_random = nx.to_numpy_array(G_random, weight='weight' if G.is_directed() else None, dtype=np.float64)

        # Compute singular values and collect them
        singular_values = np.linalg.svd(adj_matrix_random, compute_uv=False)
        singular_values_list.append(singular_values)

    # Compute mean of singular values across all random graphs
    mean_singular_values = np.mean(singular_values_list, axis=0)
    return mean_singular_values

# Step 3: Generate random graphs and compute mean singular values (defaulting to 100 random graphs)
mean_singular_values = generate_random_graph_svd(G, num_random_graphs=10)

#. Step 4: Normalize mean singular values
max_mean_singular_value = np.max(mean_singular_values)
norm_mean_singular_values = mean_singular_values/max_mean_singular_value

In [ ]:
# Plotting
fig, axs = plt.subplots(1, 1, figsize=(5,5))
# Plot mean scree plot of random graphs
axs.plot(range(1, len(mean_singular_values) + 1), norm_mean_singular_values, marker='s', linestyle='--', color='r', label='Mean of Random Graphs')
# Plot original graph's scree plot
axs.plot(range(1, len(singular_values) + 1), norm_singular_values, marker='o', linestyle='-', color='b', label='Original Graph')
axs.set_title('Normalized Scree Plot Comparison')
axs.set_xlabel('Singular Value ID')
axs.set_ylabel('Normalized Singular Value')
axs.legend()
plt.show()

<font color="magenta">**Exercise**</font>
To better understand the role of the largest singular values, compare the normalized singular values of `low_modularity_graph`and `high_modularity_graph`.

We now compute some effective ranks and show that they are all much smaller than the actual rank.

In [ ]:
def computeRank(singularValues, tolerance=1e-13):
    return len(singularValues[singularValues > tolerance])


def computeERank(singularValues, tolerance=1e-13):
    """Effective rank based on the definition using spectral entropy
     (https://ieeexplore.ieee.org/document/7098875
      and doi:10.1186/1745-6150-2-2). """
    # We use the convention 0*log(0)=0 so we remove the zero singular values
    singularValues = singularValues[singularValues > tolerance]
    normalizedSingularValues = singularValues / np.sum(singularValues)
    return np.exp(-np.sum(normalizedSingularValues
                          * np.log(normalizedSingularValues)))


def findEffectiveRankElbow(singularValues):
    """Effective rank based on the elbow method."""
    # Coordinates of the diagonal line y = 1 - x  using ax + by + c = 0.
    a, b, c = 1, 1, -1

    # Define normalized axis with first SV at (0,1) and last SV at (1,0).
    x = np.linspace(0, 1, num=len(singularValues))
    y = (singularValues - np.min(singularValues)) /\
        (np.max(singularValues) - np.min(singularValues))

    # See https://en.wikipedia.org/wiki/Distance_from_a_point_to_a_line
    # Line_defined_by_an_equation
    # Distance between the diagonal line y = 1 - x, passing through the
    # largest and the smallest singular value, and the position (x_i, y_i)
    # of the i-th singular value
    distanceToDiagonal = np.abs(a*x + b*y + c) / np.sqrt(a**2 + b**2)

    # Returns the index of the largest distance (rank must be larger than 0).
    elbowPosition = np.argmax(distanceToDiagonal) + 1  # + 1 for indices
    return elbowPosition - 1
    # - 1 to have the effective rank/nb of significant singvals


def computeEffectiveRankEnergyRatio(singularValues, threshold=0.9):
    """Effective rank based on the energy ratio."""
    normalizedCumulSquaredSingularValues = np.cumsum(np.square(singularValues))
    normalizedCumulSquaredSingularValues /= \
        normalizedCumulSquaredSingularValues[-1]
    # Below, this is the min of the argmax. See the note in the documentation
    # of np.argmax: "In case of multiple occurrences of the maximum values,
    # the indices corresponding to the first occurrence are returned."
    return np.argmax(normalizedCumulSquaredSingularValues > threshold) + 1


def computeStableRank(singularValues):
    return np.sum(singularValues*singularValues) / np.max(singularValues)**2


def computeNuclearRank(singularValues):
    return np.sum(singularValues) / np.max(singularValues)



# Compute and print various effective ranks
print("Rank:", computeRank(singular_values))
print("Effective Rank (Entropy):", computeERank(singular_values))
print("Effective Rank (Energy Ratio):", computeEffectiveRankEnergyRatio(singular_values))
print("Effective Rank (Elbow):", findEffectiveRankElbow(singular_values))
print("Nuclear Rank:", computeNuclearRank(singular_values))
print("Stable Rank:", computeStableRank(singular_values))

#### <font color="magenta">**3.5.6 Exercise**</font>
Perform all the previous analyses for the structural connectomes of the mouse and the larval Drosophila.

##### **Mesoscale SC of the mouse brain**

The mesoscopic connectome of the mouse, as mapped by the Allen Institute and reported by ([Oh et al. 2014](https://doi.org/10.1038/nature13186)), provides a detailed and high-resolution map of neural connections. It features 426 nodes (or 213 nodes, representing symmetric pairs of brain regions) and 8820 directed edges, highlighting both low-degree and high-degree nodes. The study found that nodes with lower degrees had higher clustering coefficients and identified a power-law relationship between these properties. Using this data, a generative model was developed based on proximal attachment and source growth principles, successfully replicating the structural features of the connectome, indicating the importance of spatial embedding and node growth in brain connectivity.

In [ ]:
import urllib.request

# URL of the text file
url = "https://github.com/VinceThi/low-rank-hypothesis-complex-systems/raw/v1.0.0/graphs/graph_data/connectomes/ABA_weight_mouse.txt"

# Load the data from the URL into a numpy array
with urllib.request.urlopen(url) as f:
    adj_matrix = np.loadtxt(f).astype(float)

# Create a directed graph from the matrix (assuming it represents connectivity)
G_mouse = nx.DiGraph(adj_matrix)

# Example: Print basic graph information
print(f"Number of nodes: {G_mouse.number_of_nodes()}")
print(f"Number of edges: {G_mouse.number_of_edges()}")
print(f"Density: {nx.density(G_mouse)}")
print(f"Weakly connected: {nx.is_weakly_connected(G_mouse)}")
print(f"Strongly connected: {nx.is_strongly_connected(G_mouse)}")

##### **Microscopic SC of the larval drosophila**

A complete brain connectome of the larva of the fruit fly Drosophila melanogaster as recently published in [this paper]( https://doi.org/10.1126/science.add9330). Nodes are neurons and edges are synaptic connections.

In [ ]:
import requests     # For making HTTP requests, used here to download the ZIP file from a URL
import zipfile      # For handling ZIP files, used here to extract contents from the downloaded ZIP file
import io           # Provides core tools for working with streams of data, used here for handling file content as a string
import pandas as pd # Provides data structures and data analysis tools, used here to work with tabular data (DataFrame)


# URL of the ZIP file
zip_url = "https://networks.skewed.de/net/fly_larva/files/fly_larva.csv.zip"

# Download the ZIP file
response = requests.get(zip_url)

# Check if the download was successful
if response.status_code == 200:
    # Extract the contents of the ZIP file
    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        # Check if 'edges.csv' exists in the ZIP file
        if 'edges.csv' in zf.namelist():
            # Open 'edges.csv' and read it into a Pandas DataFrame
            with zf.open('edges.csv') as csv_file:
                # Read CSV content as string
                csv_content = csv_file.read().decode('utf-8')

                # Split into lines
                lines = csv_content.splitlines()

                # Remove '# ' from the beginning of the first line
                if lines[0].startswith('# '):
                    lines[0] = lines[0][2:]

                # Join lines back into a single string
                csv_content_processed = '\n'.join(lines)

                # Read processed CSV content into DataFrame
                df_edges = pd.read_csv(io.StringIO(csv_content_processed), skipinitialspace=True)

                # Rename columns to remove leading spaces or characters
                df_edges.rename(columns={'# source': 'source', ' target': 'target', ' value': 'value'}, inplace=True)

        else:
            print("edges.csv not found in the ZIP file.")
else:
    print(f"Failed to download ZIP file. Status code: {response.status_code}")

# Display the DataFrame to verify
df_edges

In [ ]:
# Create a directed graph (DiGraph)
G_drosophila = nx.DiGraph()

# Add edges from DataFrame
for _, row in df_edges.iterrows():
    source = int(row['source'])  # Convert to int assuming nodes are represented as integers
    target = int(row['target'])  # Convert to int assuming nodes are represented as integers
    weight = float(row['count'])  # Convert to float assuming weights are represented as floats

    G_drosophila.add_edge(source, target, weight=weight)

# Count nodes and edges, measure density
num_nodes = G_drosophila.number_of_nodes()
num_edges = G_drosophila.number_of_edges()
print(f"Number of nodes: {num_nodes}")
print(f"Number of edges: {num_edges}")


## <font color="blue"> **4. Functional Connectivity** </font>

Functional connectomes are maps that detail how different parts of the brain interact and coordinate activity, providing a dynamic framework for understanding brain function and communication.

### **4.1 Relationship with Graphs**
Functional connectomes are commonly represented as **graphs** in the field of network neuroscience. In this representation:

- **Vertices** (or nodes) represent the functional components of the brain, such as individual neurons, clusters of neurons, or whole brain regions, depending on the scale of the connectome.
- **Edges** (or links) represent the functional connections between these nodes, typically assessed by measures of statistical dependence, such as **Pearson correlation**. Pearson correlation quantifies the degree to which the activity patterns of two nodes co-vary over time, indicating the strength of their functional connectivity.

#### **Functional connectome extraction**

To extract a **functional connectome (FC)** from time series data matrix $ N\times T $, where $N$ is the number of nodes and $T$ is the number of time steps, Pearson correlation coefficients are computed between each pair of nodes. These coefficients form an $N\times N$ matrix that quantifies the strength of functional connectivity between nodes, providing insights into network interactions over time. Visualization and analysis of this connectome help characterize network dynamics and organization in complex systems like brain networks.

To assess the **structure-function coupling**, we can calculate the Pearson correlation between the vectorized forms of the structural connectivity (SC) and functional connectivity (FC) matrices. This correlation quantifies how closely the anatomical connections align with the dynamic functional interactions within the network.

### **4.2 Functional dataset from Paul De Koninck's Lab**

#### **4.2.1 Experimental Setup**

Resonant-scanning two-photon microscopy was used to capture brain-wide neuronal activity at single-cell resolution in transgenic zebrafish larvae (Tg(elavl3:H2B-GCaMP6s)), with **22 larvae** aged 5-7 days post-fertilization (dpf). Larvae were immobilized in low-melting point agarose, and tail movements were tracked by a high-speed infrared camera. During the experiment, larvae were exposed to **static whole-field illumination for 10 minutes**, followed by **abrupt dark transitions (dark flashes) over 3 minutes**.

A piezo-driven objective rapidly acquired nuclear signals across 21 imaging planes, covering the entire larval brain except the most ventral neuronal populations. Imaging volumes were registered to the mapZebrain brain atlas using ANTs registration, transforming the coordinates of all neuronal nuclei into the atlas coordinate system. Neurons were assigned to one of **70 anatomical brain regions**. Five regions were excluded due to inconsistent sampling or absence of labeled nuclei. In total, the activity of approximately 54,464 neurons across 65 brain regions was recorded, representing roughly half of the neuronal population at this developmental stage.

#### **4.2.2 Dataset: Activity of 70 Brain Regions in 22 Larval Zebrafish**

The time-series data of all individual neurons were first baseline-normalized ($\Delta F / F_0$) and then averaged across 70 anatomically defined brain regions. Because this regional averaging pools together diverse local neural activity, the baseline signals may settle at a non-zero mean value.

#### 📝 **Key Experimental Notes:**

1. **Visual Stimulation Timeline:** The sensory dark flash stimulation begins exactly at $t = 660$ (time steps). To isolate **spontaneous activity**, analyze the segment before $t = 600$. To investigate **visually-evoked correlations**, slice the data tracking frames *after* $t = 600$.
2. **Under-Sampling & NaN Management:** Because some larvae do not completely sample all 70 regions, missing data rows will appear as zeros or NaNs in individual matrices. When averaging functional connectivity (FC) networks across the entire 22-fish cohort, always use `np.nanmean`. This ensures each pixel calculation is dynamically adjusted by a denominator reflecting only the valid samples for that pair of regions.
3. **Low-Frequency Filtering:** We recommend applying a 1D temporal Gaussian filter (`sigma=2`) to the raw arrays. This low-pass filter suppresses high-frequency noise and emphasizes slow, multi-second calcium fluctuations, which are essential for uncovering functional macroscale circuits.


##### 💻 **Task:**

Use Gemini to write a clean data-loading and temporal smoothing script for the regional zebrafish dataset using the online repository.

To ensure the script functions perfectly within Google Colab, copy and paste the detailed technical blueprint below into your prompt:

```text
Write a Google Colab Python script to download, load, and smooth a regional brain time-series array directly from GitHub.

The script must fulfill the following architectural requirements:
1. Online Data Retrieval: Use `urllib.request.urlretrieve` to download the target dataset from the raw GitHub URL: '[https://github.com/pdesrosiers/brain-connectivity-and-dynamics/raw/main/data/zebrafish/timeseries_regions_fish6.npy](https://github.com/pdesrosiers/brain-connectivity-and-dynamics/raw/main/data/zebrafish/timeseries_regions_fish6.npy)' and save it locally in the virtual environment as 'timeseries_regions_fish6.npy'.
2. Data Ingestion: Load the downloaded file into a NumPy array variable named `timeseries6` using `np.load`.
3. Dimensionality Sanity Check: Print the shape of the loaded array to the console to verify its structure.
4. Temporal Gaussian Smoothing: Import `gaussian_filter1d` from `scipy.ndimage`. Apply a one-dimensional Gaussian filter to the time series with a smoothing parameter of `sigma=1`, ensuring the filter runs strictly along the time axis (axis 1). Save the finalized matrix as `smoothed_timeseries6`.

Once a dataset is cleaned and smoothed, the next critical step is selecting a visualization framework that exposes macroscale temporal patterns across all 70 brain regions simultaneously. We will explore two powerful approaches:

1. **The Regional Heatmap:** Flattens the entire $(\text{Regions} \times \text{Time})$ matrix into a 2D pixel grid. It is highly effective for spotting abrupt, global network transitions—such as the exact moment visual stimulation begins—by tracking sudden vertical shifts in color intensity across the rows.
2. **The Ridgeline Plot (Stacked Joyplot):** Stacks individual line traces vertically with a controlled spatial offset and an opaque fill background. This representation maintains the continuous, wave-like organic shapes of individual regional calcium influxes without letting the lines tangle into an unreadable mess.

---

##### 💻 **Main Task:**

Use Gemini to generate a Python script that renders a high-contrast regional heatmap from your `smoothed_timeseries6` matrix.

To ensure proper layout alignment, copy and paste the detailed technical blueprint below into your prompt:

```text
Write a  Python script to plot a regional activity heatmap using Matplotlib.

The script must fulfill the following architectural requirements:
1. Matrix Display: Use `plt.imshow` to plot the `smoothed_timeseries6` matrix. Symmetrical rows must be inverted (`smoothed_timeseries6[::-1]`) so Region 1 is positioned at the top of the canvas framework.
2. Interface Limits: Set the colormap to 'coolwarm', scale the dynamic range strictly between `vmin=-1` and `vmax=1.5`, and force `aspect='auto'`.
3. Sensory Phase Annotations: Draw a black vertical dashed line (`linestyle='--'`, `linewidth=2`) at time step 600. Add bold text annotations labeling the area before frame 600 as 'Spontaneous Activity' and the area after as 'Visual Stimulation'.
4. Axis Alignment: Label the axes as 'Time step' and 'Region ID'. Configure the y-ticks to step by intervals of 10, mapping the custom labels from 70 down to 1 to match the inverted array. Add a colorbar tracking the $\Delta F / F_0$ scale.

##### 🚀 **Optional Exercise: Engineering a Ridgeline Plot**
For advanced students looking to master custom data visualization, open a new scratchpad cell and challenge Gemini to construct a fully custom, publication-quality Ridgeline Plot using manual line offsets.

```Plaintext
Write a Matplotlib script to build a stacked ridgeline plot from the `smoothed_timeseries6` matrix.

The script must fulfill the following technical constraints:
1. Spatial Stacking Loop: Iterate backward through the rows of the matrix. For each regional trace, apply a vertical multiplier offset (`i * 1.1`) to lift the line up the canvas.
2. Depth Management (Z-Order): Plot each trace as a thin black line (`linewidth=0.8`). Use `plt.fill_between` to mask the empty area directly underneath each curve with a solid white background. Carefully calibrate the `zorder` of both the line and the fill dynamically so that higher-indexed regions sit cleanly behind lower-indexed regions without clipping artifacts.
3. Timeline Anchoring: Draw a prominent vertical dashed indicator line at time step 600 with a high `zorder` to ensure it passes completely through all stacked mountains.
4. Clean Layout Aesthetics: Remove the top, right, and left bounding borders (spines). Displace the bottom x-axis downward (`spines['bottom'].set_position(('outward', 10))`) to create a floating, minimalist structural style. Configure explicit canvas boundaries using `plt.ylim` and `plt.xlim`.
```

---

### **4.2.3 Extraction of Functional Connectivity (FC)**

A single structural wiring diagram can support many different functional configurations. To see this in action, we will separate our low-pass filtered time series into two distinct temporal epochs: **spontaneous activity** (the quiet resting period before $t = 600$) and **visually-evoked activity** (during the dark flash stimulation after $t = 600$).

By computing the pairwise Pearson correlation matrix for each epoch independently, we can extract two distinct functional connectomes from the exact same brain regions. We will then plot them side-by-side with our structural baseline to visually contrast how sensory input reconfigures whole-brain functional topology.

*Note on Numerical Stability:* Because some brain regions might be completely silent in individual fish, their variance could drop to zero. To prevent mathematical division-by-zero errors (`NaN` rows) when calculating correlation coefficients, it is standard practice to inject a microscopic amount of random Gaussian noise into completely constant rows.

---

##### 💻 **Task:**

Use Gemini to write a Python script that slices your data by temporal epoch, handles numerical stability, and displays a three-panel comparative dashboard.

To guide the AI, copy and paste this conceptual blueprint into your prompt:

```text
Write a Python script to extract and compare functional connectivity matrices.

The script must fulfill the following goals:
1. Stability: Identify any completely flat rows in `smoothed_timeseries6` and add microscopic random Gaussian noise to prevent division-by-zero errors in correlations.
2. Slicing: Split the stabilized matrix into two distinct epochs along the time axis: spontaneous activity (frames before 600) and visually stimulated activity (frames 600 to the end).
3. Connectomics: Compute the pairwise Pearson correlation matrix for both temporal epochs using NumPy.
4. Visualization: Create a 1x3 subplot layout comparing the three network states side-by-side:
   - Panel 1: The structural baseline (`adjacency_matrix`)
   - Panel 2: The Spontaneous Functional Connectome
   - Panel 3: The Visually Stimulated Functional Connectome
5. Plot Details: Display all three matrices using the 'coolwarm' colormap with color limits locked between -1 and 1. Add a colorbar to each individual panel.

#### **4.2.4 Structure-Function Relationship**

#### **4.2.4 Structure-Function Relationship**

We will now investigate the relationship between structural connectivity (SC) and functional connectivity (FC) at the mesoscale.

To quantify **structure–function coupling**, we evaluate how well the physical wiring of the anatomical matrix aligns with the statistical dependencies of our two functional matrices. Because connectivity matrices are symmetric, we isolate only the unique, non-redundant links by extracting their upper triangles. This step transforms each $N \times N$ matrix into a flattened vector of $N(N-1)/2$ elements. We then compute the Pearson correlation coefficient ($R$) between the structural vector and each functional vector; a higher correlation indicates that stronger anatomical pathways directly constrain and shape stronger functional interactions.

---

##### 💻 **Task:**

Use Gemini to write a Python script that flattens your connectivity matrices using the provided helper function, calculates the structure-function coupling coefficients, and visualizes the relationships using scatter plots.

First, make sure you execute this helper function in your notebook:
```python
# Helper function to extract upper triangle values excluding the diagonal
def get_upper_triangle(matrix):
    return matrix[np.triu_indices_from(matrix, k=1)]
```
Then, copy and paste this conceptual blueprint into your prompt for Gemini:

```text
Write a Python script to calculate and visualize structure-function coupling.

The script must fulfill the following goals:
1. Vectorization: Use the provided `get_upper_triangle` function to convert `adjacency_matrix`, `spontaneous_FC`, and `stimulated_FC` into flattened 1D vectors.
2. Quantification: Compute the Pearson correlation coefficient ($R$) between the structural vector and the spontaneous functional vector, and then between the structural vector and the stimulated functional vector. Print both coupling scores to the console.
3. Visualization: Generate a 1x2 Matplotlib subplot layout containing two scatter plots:
   - Panel 1: Structural Vector (x-axis) vs. Spontaneous Functional Vector (y-axis).
   - Panel 2: Structural Vector (x-axis) vs. Stimulated Functional Vector (y-axis).
4. Plot Details: Add a linear trendline (regression line) to both scatter plots to highlight the relationship. Label the axes clearly, include the calculated coupling score ($R$) in each panel's title, and adjust the transparency (`alpha`) of the scatter points to handle overlapping data nicely.
```

---

In [ ]:
def get_upper_triangle(matrix):
    return matrix[np.triu_indices_from(matrix, k=1)]

#### **4.2.5 Topology-Distance Relationship**

Brain networks are physically embedded within the finite geometry of an organism's skull. Because maintaining long-range axonal projections requires significant metabolic energy, spatial network topology typically obeys a strong biophysical constraint: nearby brain regions exhibit higher connectivity, while the strength of interaction decreases as a function of anatomical distance. We will now test this hypothesis by evaluating how the physical Euclidean distance between brain regions shapes both spontaneous and visually stimulated functional interactions.

---

##### 💻 **Task:**

Use Gemini to write a Python script that calculates the correlation between anatomical distance and functional coupling, and visualizes the relationship using scatter plots.

To guide the AI, copy and paste this minimal blueprint into your prompt:

```text
Write a Python script to evaluate the relationship between anatomical distance and functional connectivity.

The script must fulfill the following goals:
1. Data Extraction: Isolate the unique, upper-triangle elements of your regional `distance_matrix`, `spontaneous_FC`, and `stimulated_FC` to create flattened vectors.
2. Quantification: Compute the Pearson correlation coefficient between the distance vector and both the spontaneous and stimulated functional vectors.
3. Visualization: Generate a 1x2 Matplotlib subplot layout displaying two scatter plots:
   - Panel 1: Euclidean Distance vs. Spontaneous Functional Connectivity.
   - Panel 2: Euclidean Distance vs. Stimulated Functional Connectivity.
4. Plot Details: Display the calculated correlation coefficient in the title of each respective panel, label the axes clearly, and adjust the point transparency (`alpha=0.6`) to manage data density.
````

---

#### 🚀 **Optional Exercise: Modeling Mesoscale Spatial Decay**

While a linear Pearson correlation gives us a quick snapshot of how connectivity decreases with distance, biophysical spatial constraints rarely follow a straight line. Instead, network coupling magnitudes typically drop off exponentially as a function of anatomical distance.

Open a new scratchpad cell and challenge Gemini to fit a non-linear exponential decay model to your regional data using the formula:

$$|FC(d)| = A e^{-d/\lambda} + C$$

Where $\lambda$ represents the **characteristic length**—the precise anatomical scale (in micrometers or matrix units) over which regional functional coupling drops off before hitting the asymptotic baseline floor ($C$).

---

##### 💻 **Task:**

Copy and paste this advanced blueprint into your prompt to guide Gemini through the non-linear optimization pipeline:

```text
Write a Python script using SciPy and Matplotlib to fit an exponential decay law to the relationship between distance and functional connectivity.

The script must fulfill the following goals:
1. Data Preparation: Take the absolute values of the upper-triangle functional vectors (`np.abs`) to capture coupling magnitudes, preventing positive and negative correlations from canceling out mathematically.
2. Curve Fitting: Define an exponential decay function `exponential_decay(d, A, lam, C)`. Use `scipy.optimize.curve_fit` to fit this model to your Distance vs. |Stimulated FC| data.
3. Evaluation: Compute the R² goodness-of-fit score for the model and print the optimal characteristic length (λ) and the baseline floor (C) to the console.
4. Visualization: Generate a clean scatter plot of the empirical distance vs. functional magnitude data points. Overlay a smooth, high-resolution curve representing the optimized exponential fit line.
5. Polishing: Highlight the exact position of the characteristic length (λ) using a vertical dotted indicator line, label the axes professionally, and display the R² score prominently in the legend.
```

## <font color="blue"> **5. Complementary References** </font>

- All built-in NetworkX functions with tutorials: [NetworkX web page](https://networkx.org/)
- A more advanced Python package for graphs is [graph-tool](https://graph-tool.skewed.de/), which is associated with the network catalog and repository [Netzschleuder](https://networks.skewed.de/), where you can find thousands of publicly available graphs derived from real data.
- The Brain Connectivity Toolbox MATLAB codebase ([Brain Connectivity Toolbox](https://sites.google.com/site/bctnet/home?authuser=0)) is widely utilized by brain imaging researchers.
   * This reference provides additional discussion and detail: [Complex network measures of brain connectivity: Uses and interpretations](https://www.sciencedirect.com/science/article/abs/pii/S105381190901074X)
   * A list of graph measures with brief descriptions and the types of compatible associated networks is available here: [List of measures](https://sites.google.com/site/bctnet/list-of-measures?authuser=0)
   * All the brain connectivity measures are implemented in Python and can be found in the following GitHub repository: [bctpy](https://github.com/aestrivex/bctpy)

- There exist dozens of methods to infer functional connectivity beyond Pearson correlation, including Granger Causality and Transfer Entropy. Most methods are now implemented in the Python package [pyspi](https://github.com/DynamicsAndNeuralSystems/pyspi)